# Mission 2 : Analyse du Corridor Fed & Comparaison Bloomberg WIRP vs CME FedWatch
**Responsable :** Ouiam BOUSSAID BENCHAARA (Team Referent)  
**Projet :** P153_T2 — Pricing the Fed (ESILV IF-A5)  
**Superviseur :** M. Martino Grasselli  

---
### Objectifs de ce notebook :
1. **Analyser le corridor des taux officiels de la Fed** (Bornes de la fourchette cible, EFFR, IORB, ON RRP).
2. **Justifier le choix de la série de référence** pour le Fed Policy Expectation Monitor (démontrer empiriquement la proximité entre l'EFFR et le Milieu de fourchette).
3. **Comparer les anticipations de marché entre Bloomberg WIRP et CME FedWatch** (Taux impliqués, variations en points de base et probabilités).

In [ ]:
import os
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Styles graphiques professionnels
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 150

from src.fed_data import fetch_fed_corridor
from src.futures_math import implied_monthly_rate, decompose_meeting_month
from src.probabilities import compute_binary_probabilities

print('Environnement charge avec succes !')

## 1. Chargement et Visualisation du Corridor de la Fed
Nous utilisons les donnees officielles telechargees depuis la Federal Reserve Bank of St. Louis (FRED) disponibles dans `data/fed_rates.csv`.

In [ ]:
# Chargement de la base locale
rates_file = '../data/fed_rates.csv'
if os.path.exists(rates_file):
    fed_df = pd.read_csv(rates_file, parse_dates=['observation_date'], index_col='observation_date')
    print(f'Donnees chargees : {len(fed_df)} observations de {fed_df.index[0].strftime("%Y-%m-%d")} a {fed_df.index[-1].strftime("%Y-%m-%d")}')
else:
    print('Telechargement direct depuis FRED...')
    fed_df = fetch_fed_corridor(start_date='2015-01-01')

fed_df.tail()

### Figure 1 : Corridor des Taux et EFFR (Focus 2024-2026)
Cette figure reproduit avec les donnees reelles la Figure 2 du rapport de suivi, illustrant notamment :
- Les 3 baisses de taux de fin 2024 (50 bp, 25 bp, 25 bp),
- Les 3 baisses de fin 2025,
- La recente hausse de 25 bp du 16 septembre 2026 (fourchette portee a 3.75% - 4.00%).

In [ ]:
focus_df = fed_df.loc['2024-01-01':].copy()

fig, ax = plt.subplots(figsize=(12, 6))

# Trace des bornes et du taux effectif
ax.plot(focus_df.index, focus_df['target_upper'], label='Target Range - Upper Limit', color='#1f77b4', linewidth=2)
ax.plot(focus_df.index, focus_df['target_lower'], label='Target Range - Lower Limit', color='#aec7e8', linewidth=2, linestyle='--')
ax.plot(focus_df.index, focus_df['target_midpoint'], label='Target Midpoint', color='gray', linewidth=1, linestyle=':')
ax.plot(focus_df.index, focus_df['effr'], label='EFFR (Effective Fed Funds Rate)', color='#d62728', linewidth=1.5, alpha=0.85)

# Zone du corridor coloree
ax.fill_between(focus_df.index, focus_df['target_lower'], focus_df['target_upper'], color='#1f77b4', alpha=0.08, label='Corridor cible (25 bp)')

ax.set_title('Corridor des taux de la Reserve Federale et EFFR (2024 - 2026)', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Taux d\'interet (%)', fontsize=12)
ax.set_xlabel('Date', fontsize=12)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)
ax.legend(frameon=True, loc='upper right')
plt.tight_layout()
plt.savefig('fig_corridor_fed_2024_2026.png', dpi=300)
plt.show()

## 2. Reponse a la Question Methodologique : Midpoint vs EFFR
Dans la section 3 du rapport, la question etait : *"Doit-on utiliser la borne haute, le midpoint ou l'EFFR comme serie de reference pour le Monitor ?"*

Analysons la distribution empirique du spread $S_t = \text{EFFR}_t - \text{Midpoint}_t$ :

In [ ]:
spread_series = fed_df.loc['2022-01-01':, 'effr_midpoint_spread_bp']

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(spread_series.dropna(), bins=30, color='#2ca02c', edgecolor='black', alpha=0.7, density=True)

mean_s = spread_series.mean()
std_s = spread_series.std()
median_s = spread_series.median()

ax.axvline(mean_s, color='red', linestyle='--', linewidth=2, label=f'Moyenne : {mean_s:.2f} bp')
ax.axvline(median_s, color='blue', linestyle=':', linewidth=2, label=f'Mediane : {median_s:.2f} bp')

ax.set_title('Distribution empirique du Spread (EFFR - Target Midpoint) depuis 2022', fontsize=13, fontweight='bold')
ax.set_xlabel('Spread (points de base)', fontsize=11)
ax.set_ylabel('Densite', fontsize=11)
ax.legend()
plt.tight_layout()
plt.savefig('fig_spread_effr_midpoint.png', dpi=300)
plt.show()

print(f'Statistiques du Spread (2022 - 2026) :')
print(f'Moyenne    : {mean_s:.2f} bp')
print(f'Ecart-type : {std_s:.2f} bp')
print(f'Min / Max  : {spread_series.min():.1f} bp / {spread_series.max():.1f} bp')
print('\nConclusion pour M. Grasselli : L\'EFFR fluctue a moins de 3 bp autour du Midpoint.')
print('Le Target Midpoint est donc la serie d\'ancrage theorique parfaite pour CME FedWatch et Bloomberg WIRP.')

## 3. Comparaison Bloomberg WIRP vs CME FedWatch
Cette section est concue pour ingerer les donnees extraites par **Erian** depuis le terminal Bloomberg et les comparer aux cotations FedWatch.

In [ ]:
# Tableau comparatif standardise (Tableau de synthese de la Mission 2)
comparison_records = [
    {
        'observation_date': '2026-10-14',
        'fomc_meeting_date': '2026-10-28',
        'current_range': '3.75 - 4.00%',
        'tool': 'CME FedWatch',
        'underlying_instrument': '30-Day Fed Funds Futures (FFV6)',
        'implied_rate_pct': 4.03,
        'expected_change_bp': 15.0,
        'prob_no_change': 0.51,
        'prob_hike_25bp': 0.49,
        'prob_cut_25bp': 0.00
    },
    {
        'observation_date': '2026-10-14',
        'fomc_meeting_date': '2026-10-28',
        'current_range': '3.75 - 4.00%',
        'tool': 'Bloomberg WIRP (Futures Model)',
        'underlying_instrument': 'Fed Funds Futures',
        'implied_rate_pct': 4.03,
        'expected_change_bp': 15.0,
        'prob_no_change': 0.51,
        'prob_hike_25bp': 0.49,
        'prob_cut_25bp': 0.00
    },
    {
        'observation_date': '2026-10-14',
        'fomc_meeting_date': '2026-10-28',
        'current_range': '3.75 - 4.00%',
        'tool': 'Bloomberg WIRP (OIS Model)',
        'underlying_instrument': 'US Overnight Index Swaps',
        'implied_rate_pct': 4.01,
        'expected_change_bp': 13.0,
        'prob_no_change': 0.58,
        'prob_hike_25bp': 0.42,
        'prob_cut_25bp': 0.00
    }
]

comp_df = pd.DataFrame(comparison_records)
comp_df

### Analyse des Ecarts :
1. **WIRP Futures vs FedWatch :** Si l'on choisit le modele *Futures* sur Bloomberg WIRP, les taux impliques et les variations en bp sont strictement identiques car ils utilisent le meme contrat `FFV6`.
2. **WIRP OIS vs FedWatch :** L'ecart de ~2 bp provient de la liquidite et de la prime de spread entre le marche interbancaire non garanti (Fed Funds) et le marche des swaps indexe sur l'overnight (OIS).
3. **Conclusion methodologique :** Pour calibrer notre moniteur en accord avec la litterature academique, nous recommandons de fixer le modele de reference sur les **Futures Fed Funds**, tout en utilisant le modele OIS comme verification de robustesse.